# Error Case 13 — 404 Not Found: Catalog Not Found

**Error:** `404 Not Found`  
**Root cause:** Catalog name typo or catalog was deleted while ETL still running  
**Fix:** Verify catalog name, recreate if deleted

## Flow
```
GET /catalog/v1/{catalog}/namespaces
  → Auth: JWT valid ✅
  → RBAC: check grants ✅
  → Lookup catalog by name in DB
  → catalog not found → 404
```

## Common production scenarios
```
1. Typo in catalog name (analytics vs analytic)
2. Catalog deleted manually, ETL still running
3. Wrong environment (prod catalog name in dev config)
4. Catalog renamed — old name still in ETL config
```

In [1]:
import sys
sys.path.insert(0, '..')
import importlib
import polaris_test_utils
importlib.reload(polaris_test_utils)
from polaris_test_utils import *

cleanup()
print('✅ Setup complete')

🧹 Cleaning up test resources...
✅ Cleanup done
✅ Setup complete


In [ ]:
# Cell 2: Get valid token first
r = get_token()
print_response(r, 'Valid token response')
valid_token = r.json()['access_token']
print(f'\n✅ Got valid token')

In [2]:
# Cell 2: Trigger 404 — typo in catalog name
WRONG_CATALOG = 'nonexistent-catalog-xyz'

r = requests.get(
    f'{BASE_CAT}/{WRONG_CATALOG}/namespaces',
    headers=h()
)
print_response(r, '404 Catalog not found (typo)')
assert r.status_code == 404
print('\n✅ 404 confirmed — catalog not found')


404 Catalog not found (typo):
  Status:     404
  Request-Id: 9887a791-517e-47dd-b249-e2806aed84a9_0000000000000000135
  Body: {
  "error": {
    "message": "Namespace does not exist: ",
    "type": "NoSuchNamespaceException",
    "code": 404
  }
}

✅ 404 confirmed — catalog not found


In [3]:
# Cell 3: Simulate — catalog deleted while ETL running
# Step 1: Create catalog
r1 = create_catalog()
assert r1.status_code == 201, r1.text
print(f'Catalog created: {TEST_CATALOG}')

# Step 2: Delete catalog (simulates operator deleting catalog)
r2 = delete_catalog()
assert r2.status_code == 204
print(f'Catalog deleted: {TEST_CATALOG}')

# Step 3: ETL tries to use deleted catalog
r3 = requests.get(
    f'{BASE_CAT}/{TEST_CATALOG}/namespaces',
    headers=h()
)
print_response(r3, '404 Catalog deleted mid-ETL')
assert r3.status_code == 404
print('\n✅ 404 confirmed — catalog deleted while ETL running')

Catalog created: test-error-catalog
Catalog deleted: test-error-catalog

404 Catalog deleted mid-ETL:
  Status:     404
  Request-Id: 9887a791-517e-47dd-b249-e2806aed84a9_0000000000000000141
  Body: {
  "error": {
    "message": "Namespace does not exist: ",
    "type": "NoSuchNamespaceException",
    "code": 404
  }
}

✅ 404 confirmed — catalog deleted while ETL running


In [4]:
# Cell 4: Find error logs
wait_for_logs(5)
print('Searching for 404 errors (last 1 min)...')
hits = search_404_errors(minutes_ago=1)
print_logs(hits)

⏳ Waiting 5s for logs to reach OpenSearch...
Searching for 404 errors (last 1 min)...
Found 10 log entries:

ℹ️  [2026-06-02T06:06:35.783Z] INFO
   logger    : io.quarkus.http.access-log
   requestId : 9887a791-517e-47dd-b249-e2806aed84a9_0000000000000000141
   realmId   : POLARIS
   message   : 192.168.194.1 - root [02/Jun/2026:06:06:35 +0000] "GET /api/catalog/v1/test-error-catalog/namespaces HTTP/1.1" 404 95

ℹ️  [2026-06-02T06:06:35.782Z] INFO
   logger    : org.apache.polaris.service.exception.IcebergExceptionMapper
   requestId : 9887a791-517e-47dd-b249-e2806aed84a9_0000000000000000141
   realmId   : POLARIS
   message   : Handling runtimeException Namespace does not exist: 

ℹ️  [2026-06-02T06:06:30.895Z] INFO
   logger    : io.quarkus.http.access-log
   requestId : 9887a791-517e-47dd-b249-e2806aed84a9_0000000000000000135
   realmId   : POLARIS
   message   : 192.168.194.1 - root [02/Jun/2026:06:06:30 +0000] "GET /api/catalog/v1/nonexistent-catalog-xyz/namespaces HTTP/1.1" 404 9

In [5]:
# Cell 5: Fix — verify catalog exists before ETL
r_check = requests.get(f'{BASE_MGMT}/catalogs/{TEST_CATALOG}', headers=h())
print(f'Catalog exists check: {r_check.status_code}')

if r_check.status_code == 404:
    print('Catalog not found — recreating...')
    r_fix = create_catalog()
    assert r_fix.status_code == 201
    print('✅ Catalog recreated')
else:
    print('✅ Catalog exists')

# Verify access works
r_verify = requests.get(
    f'{BASE_CAT}/{TEST_CATALOG}/namespaces',
    headers=h()
)
print(f'\nVerify access: {r_verify.status_code}')
assert r_verify.status_code == 200

cleanup()
print('\n=== Summary ===')
print('Error:    404 Not Found')
print('Cause:    Catalog deleted or name typo')
print('Detect:   404 with NoSuchNamespaceException or similar')
print('Fix:      GET /catalogs/{name} to verify existence before ETL')
print('Pattern:  Add catalog existence check at ETL startup')
print('Alert:    Set OpenSearch alert on 404 error spike')

Catalog exists check: 404
Catalog not found — recreating...
✅ Catalog recreated

Verify access: 200
🧹 Cleaning up test resources...
✅ Cleanup done

=== Summary ===
Error:    404 Not Found
Cause:    Catalog deleted or name typo
Detect:   404 with NoSuchNamespaceException or similar
Fix:      GET /catalogs/{name} to verify existence before ETL
Pattern:  Add catalog existence check at ETL startup
Alert:    Set OpenSearch alert on 404 error spike
